#  Data sharing
* OpenMP is a shared memory programming model, which means threads see updates among each other
* That is, when a thread updates a variable _x_ that is then read by another, the reader thread will see the updated value
* This is the default behavior of OpenMP ([Data Environment](https://www.openmp.org/spec-html/5.2/openmpse27.html#x135-5430002.19) in OpenMP spec)
* It is not always convenient, however
* `#pragma omp parallel` can thus specify whether local variables in the scope (i.e., defined outside the statement) are privatized (i.e., made private to each thread)

# <font color="green"> Problem 1 :  Observe the effect of privatization</font>
* Execute the following and make sense of the output
* Add the `private(x)` clause and observe the difference


In [ ]:
%%writefile omp_private.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
  /* add private(x) clause below and see the difference */
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_private.c -o omp_private
# nvc -mp omp_private.c -o omp_private

In [ ]:
%%bash
OMP_NUM_THREADS=10 ./omp_private

* `private(x)` essentially ignores the original variable `x` defined outside the parallel region and behaves as if a variable of the same name is defined by each thread
* `firstprivate(x)` is like `private(x)`, except that `x` of each thread is initialized by the value of `x` just before entering the parallel region

# <font color="green"> Problem 2 :  Observe the effect of `private` and `firstprivate`</font>
* Execute the following and observe the output
* Add the `private(x)` clause and execute it
* Add the `firstprivate(x)` clause and execute it
* Make sense of the differences


In [ ]:
%%writefile omp_firstprivate.c
#include <stdio.h>
#include <unistd.h>
#include <omp.h>

int main() {
  int x = 123;
  printf("before : x = %d\n", x);
  /* add private(x)/firstprivate(x) clause and see the difference */
#pragma omp parallel
  {
    int id = omp_get_thread_num();
    x++;
    printf("thread %d : x = %d\n", id, x);
  }
  printf("after : x = %d\n", x);
  return 0;
}

In [ ]:
%%bash
clang -fopenmp omp_firstprivate.c -o omp_firstprivate
# nvc -mp omp_firstprivate.c -o omp_firstprivate

In [ ]:
%%bash
OMP_NUM_THREADS=3 ./omp_firstprivate